In [57]:
from openai import OpenAI
import os
from IPython.display import Markdown, display
import dotenv
import json
from PyPDF2 import PdfReader
import requests
from dotenv import load_dotenv
import gradio as gr

In [58]:
load_dotenv(override=True)
openai = OpenAI()

In [59]:
pushover_user = os.getenv("PUSHOVER_USER")
print(f"Pushover user: {pushover_user}")
pushover_token = os.getenv("PUSHOVER_TOKEN")
print(f"Pushover token: {pushover_token}")
pushover_url = "https://api.pushover.net/1/messages.json"

Pushover user: uiyyechy81sokwitscr5z5jgy52ddg
Pushover token: a313qczbheno5ami483fpbyk4xsorn


In [60]:
def push(message):
    print(f"Pushing notification: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    response = requests.post(pushover_url, data=payload)
    print(f"Status: {response.status_code}")
    print(f"Response: {response.json()}")

In [61]:
push("Hello from your agent!")

Pushing notification: Hello from your agent!
Status: 200
Response: {'status': 1, 'request': 'f5488d79-ec51-4abf-a132-bac9ee717829'}


In [62]:
def record_user_details(email,name="Name not provided" , notes="not provided"):
    push(f"Recording interest from {name} with email {email} and notes {notes} ")
    return {"recorded" : "ok"}

In [63]:
def record_unknown_question(question):
    push(f"Recording unknown question: {question} asked that i couldn't answer")
    return {"recorded" : "ok"}

In [64]:
record_user_details_json = {
    "name" : "record_user_details",
    "description" : "use this tool to record user details when they express interest in the product or service. This will help us follow up with them and provide more information about our offerings. The email field is required, while the name and notes fields are optional.",
    "parameters" : {
        "type" : "object",
        "properties" : {
            "email" : {
                "type" : "string",
                "description" : "The email address of the user expressing interest. This field is required."
            },
            "name" : {
                "type" : "string",
                "description" : "The name of the user expressing interest. This field is optional."
            },
            "notes" : {
                "type" : "string",
                "description" : "Any additional notes or information about the user's interest. This field is optional."
            }
        },
        "required" : ["email"],
        "additionalProperties" : False
    }
}

In [65]:
record_unknown_question_json = {
    "name" : "record_unknown_question",
    "description" : "Always use this tool to record any questions that you are unable to answer. This will help us identify areas where we may need to improve our knowledge base or provide additional training.",
    "parameters" : {
        "type" : "object",
        "properties" : {
            "question" : {
                "type" : "string",
                "description" : "The question that you were unable to answer. This field is required."
            }
        },
        "required" : ["question"],
        "additionalProperties" : False
    }
}

In [66]:
tools = [{"type" : "function" , "function" : record_user_details_json},
         {"type" : "function" , "function" : record_unknown_question_json}]


In [67]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Handling tool call for {tool_name} with arguments {arguments}", flush=True)


        if tool_name == "record_user_details":
            result = record_user_details(**arguments)
        elif tool_name == "record_unknown_question":
            result = record_unknown_question(**arguments)

        results.append({"role" : "tool" , "content" : json.dumps(result) , "tool_call_id" : tool_call.id})

    return results

In [68]:
globals()["record_unknown_question"]("this is a really hard question")

Pushing notification: Recording unknown question: this is a really hard question asked that i couldn't answer
Status: 200
Response: {'status': 1, 'request': '69354ea4-ae3e-4253-ae72-94e59930b70a'}


{'recorded': 'ok'}

In [69]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Handling tool call for {tool_name} with arguments {arguments}", flush=True)

        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role" : "tool" , "content" : json.dumps(result) , "tool_call_id" : tool_call.id})
    return results    

In [70]:
reader = PdfReader("me/Abhishek_Mittal_Resume.pdf")
linkedin = ""

for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

with open("me/Abhishek.txt", "r", encoding="utf-8") as f:
    summary = f.read()

name = "Abhishek Mittal"

In [71]:
system_prompt = f"""You are a professional AI representative acting as {name}. Your role is to engage with visitors, answer questions about {name}'s background, experience, and skills, and identify potential professional connections.

## Your Identity
You are {name}. Respond in first person, maintaining a confident, professional, and approachable tone at all times.

## Knowledge Base
You have access to the following information about {name}. Only answer questions based on this information — do not fabricate or infer details not explicitly present.

### Resume
{linkedin}

### Personal Summary
{summary}

if you don't know the answer to a question, say you don't know and use the record_unknown_question tool to log it for future improvement. If a visitor expresses interest in connecting, collaborating, or learning more, use the record_user_details tool to capture their email and any relevant notes.
if user expresses interest in connecting, collaborating, or learning more, use the record_user_details tool to capture their email and any relevant notes. Always use the record_unknown_question tool to log any questions that you are unable to answer. This will help us identify areas where we may need to improve our knowledge base or provide additional training.
if the user is engaging in discussion, try to steer them towards getting in touch via email or connecting on LinkedIn to continue the conversation and build a professional relationship.

## Behavioral Guidelines
- Speak in first person as {name} — never break character
- Be concise, professional, and helpful in all responses
- If asked something you cannot answer from the provided information, be transparent: say you're not sure, then use the `record_unknown_question` tool to log it
- If a visitor expresses interest in connecting, collaborating, or learning more, use the `record_user_details` tool to capture their email and any relevant notes
- Do not speculate, exaggerate, or provide information not supported by the resume or summary
- Keep responses focused and relevant to professional topics

## Tool Usage Rules (STRICT — NO EXCEPTIONS)
- If ANY question cannot be answered directly from the Resume or Personal Summary above, 
  you MUST call `record_unknown_question` BEFORE responding. No exceptions.
- If the user shares an email address in ANY message for ANY reason, 
  you MUST immediately call `record_user_details` with that email. No exceptions.
- These tools are mandatory actions, not suggestions. Always use them as specified to ensure accurate logging and follow-up.

"""

In [72]:
def chat(message, history):
    messages = [{"role" : "system" , "content" : system_prompt}] + history + [{"role" : "user" , "content" : message}]
    done = False
    while not done:
        response = openai.chat.completions.create(model="gpt-4o", messages=messages, tools=tools)
        finish_response = response.choices[0].finish_reason
        if finish_response == "tool_calls":
            assistant_message = response.choices[0].message
            tool_calls = assistant_message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(assistant_message)
            messages.extend(results)
        else:
            done = True

    return response.choices[0].message.content        

In [73]:
gr.ChatInterface(chat , type="messages").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Handling tool call for record_unknown_question with arguments {'question': 'who is your favorite person?'}
Pushing notification: Recording unknown question: who is your favorite person? asked that i couldn't answer
Status: 200
Response: {'status': 1, 'request': '53cd9413-25d5-4de8-ab14-1814a8f0d264'}
Handling tool call for record_user_details with arguments {'email': 'em@gmail.com'}
Pushing notification: Recording interest from Name not provided with email em@gmail.com and notes not provided 
Status: 200
Response: {'status': 1, 'request': '48a064ea-ebe0-49eb-98f2-c2614ec144a5'}


Exception in callback _ProactorBasePipeTransport._call_connection_lost(None)
handle: <Handle _ProactorBasePipeTransport._call_connection_lost(None)>
Traceback (most recent call last):
  File "C:\Users\mitta\AppData\Roaming\uv\python\cpython-3.12.12-windows-x86_64-none\Lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
  File "C:\Users\mitta\AppData\Roaming\uv\python\cpython-3.12.12-windows-x86_64-none\Lib\asyncio\proactor_events.py", line 165, in _call_connection_lost
    self._sock.shutdown(socket.SHUT_RDWR)
ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host


# For the dev Production